# 04 — Model Evaluation
Offline evaluation of all models using Hit Rate, NDCG, MRR, Precision, Recall, Coverage, and Novelty.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import pandas as pd
import scipy.sparse as sp
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.dirname(os.getcwd())
PROCESSED    = f"{PROJECT_ROOT}/data/processed"
MODELS_DIR   = f"{PROJECT_ROOT}/outputs/models"
REPORTS_DIR  = f"{PROJECT_ROOT}/outputs/reports"
FIGURES_DIR  = f"{PROJECT_ROOT}/outputs/figures"
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Setup complete.")

## 1. Load Models & Test Data

In [ ]:
from models.collaborative_filtering import ALSRecommender
from models.content_based import ContentBasedRecommender
from models.session_based import SessionBasedRecommender
from models.hybrid import HybridRecommender

interactions = pd.read_parquet(f"{PROCESSED}/interactions.parquet")
sequences    = pd.read_parquet(f"{PROCESSED}/session_sequences.parquet")

# Temporal test split: last 20% of sessions by sequence order
cutoff = int(len(sequences) * 0.8)
test_sequences = sequences.iloc[cutoff:].reset_index(drop=True)

# Load trained models
als    = ALSRecommender.load(f"{MODELS_DIR}/als_model.pkl")
cb     = ContentBasedRecommender.load(f"{MODELS_DIR}/content_based_model.pkl")
sb     = SessionBasedRecommender.load(f"{MODELS_DIR}/session_based_model.pkl")
hybrid = HybridRecommender(cf_model=als, cb_model=cb, sb_model=sb)

item_popularity = interactions.groupby("itemid")["score"].sum().to_dict()
catalogue_size  = interactions["itemid"].nunique()

print(f"Test sessions  : {len(test_sequences):,}")
print(f"Catalogue size : {catalogue_size:,}")

## 2. Evaluate Each Model

In [ ]:
from training.evaluate import evaluate_all_models

all_results = {}
models_to_eval = {
    "Session-Based (GRU4Rec)": sb,
    "Content-Based":           cb,
    "Hybrid":                  hybrid,
}

for model_name, model in models_to_eval.items():
    print(f"\nEvaluating: {model_name}")
    results = evaluate_all_models(
        model=model,
        test_sequences=test_sequences,
        interactions=interactions,
        catalogue_size=catalogue_size,
        item_popularity=item_popularity,
        k_values=[5, 10],
    )
    all_results[model_name] = results
    display(results)

## 3. Comparison Charts

In [ ]:
metrics = ["hit_rate", "ndcg", "mrr", "precision", "recall", "coverage", "novelty"]
k = 10

comparison = pd.DataFrame({
    name: res.loc[k, metrics]
    for name, res in all_results.items()
}).T

display(comparison)

fig, axes = plt.subplots(1, len(metrics), figsize=(18, 4))
for ax, metric in zip(axes, metrics):
    comparison[metric].plot(kind="bar", ax=ax, color=["#4C72B0","#55A868","#DD8452"])
    ax.set_title(f"{metric} @{k}")
    ax.set_xticklabels(comparison.index, rotation=30, ha="right")
    ax.set_ylabel("")
plt.suptitle(f"Model Comparison @ K={k}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/model_comparison_k{k}.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Export Evaluation Report

In [ ]:
report_path = f"{REPORTS_DIR}/evaluation_report.csv"
comparison.to_csv(report_path)
print(f"Evaluation report saved to {report_path}")